# slum population

In [ ]:
# ============================================================
# Sub-Saharan Africa slum population
# ============================================================

from pathlib import Path
import math
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from rasterio.features import geometry_mask

tif_path = Path(r"..\data\slum pop\subsaharan_africa_slum_population.tif")
shp_path = Path(r"..\data\overall\admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp")
country_shp_path = Path(r"..\data\country\country_SSA_HE2.shp")

out_path = Path(r"figures/slum_population_map.png")


mpl.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 13,
})

BACKGROUND_COLOR = "#FFFFFF"

SLUM_COLORS = [
    "#F1EEF2",
    "#DDD1DF",
    "#C1AAC3",
    "#98789D",
    "#694E70",
]


def add_scalebar(
    ax,
    length_km=1000,
    location=(0.28, 0.055),
    linewidth=1.4
):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)

    lat0 = y0
    km_per_deg_lon = 111.32 * math.cos(math.radians(lat0))

    if km_per_deg_lon <= 0:
        km_per_deg_lon = 111.32

    length_deg = length_km / km_per_deg_lon
    x1 = x0 + length_deg

    right_margin = xmax - 0.03 * (xmax - xmin)

    if x1 > right_margin:
        x1 = right_margin
        x0 = x1 - length_deg

    ax.plot(
        [x0, x1],
        [y0, y0],
        color="#333333",
        linewidth=linewidth,
        solid_capstyle="butt",
        zorder=20
    )

    ax.text(
        (x0 + x1) / 2,
        y0 + 0.85,
        f"{length_km:,} km",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color="#333333",
        zorder=21
    )

    x_center_frac = ((x0 + x1) / 2 - xmin) / (xmax - xmin)
    y_frac = (y0 - ymin) / (ymax - ymin)

    return x_center_frac, y_frac


def add_north_arrow(
    ax,
    location=(0.35, 0.14),
    arrow_length=0.035
):
    x, y = location

    ax.annotate(
        "",
        xy=(x, y + arrow_length),
        xytext=(x, y),
        xycoords=ax.transAxes,
        arrowprops=dict(
            arrowstyle="-|>",
            lw=1.0,
            color="#333333",
            mutation_scale=10
        ),
        zorder=30
    )

    ax.text(
        x,
        y + arrow_length + 0.008,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color="#333333",
        zorder=31
    )


with rasterio.open(tif_path) as src:

    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Size:", src.width, "x", src.height)
    print("Bands:", src.count)

    data = src.read(1, masked=True)

    xmin = src.bounds.left
    ymin = src.bounds.bottom
    xmax = src.bounds.right
    ymax = src.bounds.top

    mean_lat = (ymin + ymax) / 2


    country_gdf = gpd.read_file(country_shp_path)

    if country_gdf.crs is not None and country_gdf.crs != src.crs:
        country_gdf = country_gdf.to_crs(src.crs)


    inside_country = geometry_mask(
        country_gdf.geometry,
        out_shape=data.shape,
        transform=src.transform,
        invert=True
    )

    data = np.ma.array(
        data,
        mask=np.ma.getmaskarray(data) | (~inside_country)
    )


    valid_values = data.compressed()
    positive_values = valid_values[valid_values > 0]

    if positive_values.size == 0:
        raise ValueError(
            "Raster contains no values greater than zero."
        )

    eps = float(positive_values.min()) * 0.999

    bounds = np.array([
        0,
        eps,
        np.quantile(positive_values, 0.25),
        np.quantile(positive_values, 0.50),
        np.quantile(positive_values, 0.75),
        np.quantile(positive_values, 1.00)
    ], dtype=float)

    bounds = np.unique(bounds)

    n_classes = len(bounds) - 1

    if n_classes < 2:
        raise ValueError(
            "Not enough unique values to construct discrete classes."
        )


    if n_classes <= len(SLUM_COLORS):
        colors = SLUM_COLORS[:n_classes]
    else:
        base_cmap = plt.get_cmap("Purples", n_classes + 2)

        colors = [
            mpl.colors.to_hex(base_cmap(i + 1))
            for i in range(n_classes)
        ]

    cmap = ListedColormap(colors)
    cmap.set_bad((1, 1, 1, 0))

    norm = BoundaryNorm(
        bounds,
        cmap.N
    )


    fig, (ax, ax_bar) = plt.subplots(
        1, 2,
        figsize=(13.2, 8.4),
        dpi=300,
        gridspec_kw={"width_ratios": [1.42, 0.88], "wspace": 0.2}
    )

    fig.patch.set_facecolor("white")
    ax.set_facecolor(BACKGROUND_COLOR)
    ax_bar.set_facecolor("white")
    ax.set_title(
        "(a) Slum population distribution",
        loc="left",
        pad=6,
        fontsize=13,
        fontweight="bold",
        color="#2E2E2E"
    )


    ax.imshow(
        data,
        cmap=cmap,
        norm=norm,
        extent=(xmin, xmax, ymin, ymax),
        origin="upper",
        interpolation="nearest",
        zorder=1
    )


    gdf = gpd.read_file(shp_path)

    if gdf.crs is not None and gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    gdf.boundary.plot(
        ax=ax,
        color="#FFFFFF",
        linewidth=0.15,
        alpha=0.50,
        zorder=3
    )


    country_gdf.boundary.plot(
        ax=ax,
        edgecolor="#555555",
        linewidth=0.40,
        alpha=0.90,
        facecolor="none",
        zorder=4
    )

    country_gdf.dissolve().boundary.plot(
        ax=ax,
        edgecolor="#303030",
        linewidth=0.65,
        facecolor="none",
        zorder=5
    )


    map_xmin, map_ymin, map_xmax, map_ymax = country_gdf.total_bounds

    x_pad = (map_xmax - map_xmin) * 0.015
    y_pad = (map_ymax - map_ymin) * 0.015

    ax.set_xlim(
        map_xmin - x_pad,
        map_xmax + x_pad
    )

    ax.set_ylim(
        map_ymin - y_pad,
        map_ymax + y_pad
    )

    map_mean_lat = (map_ymin + map_ymax) / 2

    ax.set_aspect(
        1 / np.cos(np.deg2rad(map_mean_lat))
    )


    scalebar_center_x, scalebar_y = add_scalebar(
        ax,
        length_km=1000,
        location=(0.28, 0.055),
        linewidth=1.4
    )

    add_north_arrow(
        ax,
        location=(
            scalebar_center_x,
            scalebar_y + 0.060
        ),
        arrow_length=0.035
    )


    ax.set_xticks([])
    ax.set_yticks([])

    ax.tick_params(
        left=False,
        right=False,
        bottom=False,
        top=False,
        labelleft=False,
        labelbottom=False
    )

    for spine in ax.spines.values():
        spine.set_visible(False)


    legend_handles = []

    for i in range(n_classes):

        lower = bounds[i]
        upper = bounds[i + 1]

        if i == 0:
            label = "0"
        else:
            label = f"{lower:,.0f}–{upper:,.0f}"

        legend_handles.append(
            Patch(
                facecolor=colors[i],
                edgecolor="none",
                label=label
            )
        )

    legend = ax.legend(
        handles=legend_handles,
        title="Slum population",
        loc="lower left",
        bbox_to_anchor=(0.015, 0.055),
        frameon=False,
        fontsize=9,
        title_fontsize=10,
        handlelength=1.5,
        handleheight=0.9,
        handletextpad=0.7,
        labelspacing=0.55,
        borderpad=0
    )

    legend.get_title().set_fontweight("semibold")
    legend.get_title().set_color("#333333")


    if "iso3" not in gdf.columns or "sl_pop" not in gdf.columns:
        raise KeyError("Fields 'iso3' and 'sl_pop' are required for the country ranking panel.")
    if "country_na" not in country_gdf.columns:
        raise KeyError("Field 'country_na' is required in the country boundary file.")

    admin_points = gdf[["iso3", "sl_pop", "geometry"]].copy()
    admin_points["geometry"] = admin_points.geometry.representative_point()
    country_joined = gpd.sjoin(
        admin_points,
        country_gdf[["country_na", "geometry"]],
        how="left",
        predicate="within"
    )

    iso3_name_fallback = {
        "COM": "Comoros",
        "STP": "S?o Tom? and Pr?ncipe",
        "SYC": "Seychelles",
    }
    country_joined["country_label"] = (
        country_joined["country_na"]
        .fillna(country_joined["iso3"].map(iso3_name_fallback))
        .fillna(country_joined["iso3"])
    )

    top_country = (
        country_joined[["country_label", "sl_pop"]]
        .dropna(subset=["country_label", "sl_pop"])
        .assign(sl_pop=lambda df: df["sl_pop"].clip(lower=0))
        .groupby("country_label", as_index=False)["sl_pop"]
        .sum()
        .sort_values("sl_pop", ascending=False)
        .head(20)
    )

    y = np.arange(len(top_country))
    pop_million = top_country["sl_pop"].to_numpy(dtype=float) / 1_000_000
    bar_colors = [
        mpl.colors.to_hex(plt.get_cmap("Purples")(0.78 - 0.44 * i / max(len(top_country) - 1, 1)))
        for i in range(len(top_country))
    ]

    ax_bar.barh(
        y,
        pop_million,
        height=0.68,
        color=bar_colors,
        edgecolor="white",
        linewidth=0.7,
        zorder=3
    )

    x_offset = max(pop_million.max(), 1) * 0.014
    for yi, val in zip(y, pop_million):
        ax_bar.text(
            val + x_offset,
            yi,
            f"{val:.1f}",
            ha="left",
            va="center",
            fontsize=6.8,
            color="#4B3E50"
        )

    ax_bar.set_title(
        "(b) Countries ranked by slum population",
        loc="left",
        pad=6,
        fontsize=13,
        fontweight="bold",
        color="#2E2E2E"
    )
    ax_bar.set_xlabel("Slum population (million)")
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels(top_country["country_label"])
    ax_bar.invert_yaxis()
    ax_bar.set_xlim(0, pop_million.max() * 1.16)
    ax_bar.grid(axis="x", color="#E0E0E0", linewidth=0.6, alpha=0.85, zorder=0)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["left"].set_color("#000000")
    ax_bar.spines["bottom"].set_color("#000000")
    ax_bar.tick_params(axis="both", colors="#000000")
    ax_bar.tick_params(axis="y", labelsize=7.8, pad=2)
    ax_bar.xaxis.label.set_color("#000000")
    ax_bar.yaxis.label.set_color("#000000")


    fig.subplots_adjust(
        left=0.018,
        right=0.985,
        bottom=0.055,
        top=0.945
    )

    out_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.05
    )

    plt.show()

# WBGT extreme heat exposure and days

In [ ]:
from pathlib import Path
import re
import math

import numpy as np
import geopandas as gpd
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch


# =====================
# 1. Paths
# =====================

folder = Path(r"../data/WBGT30_days").resolve()

shp_path = Path(
    r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
).resolve()

country_shp_path = Path(
    r"../data/country/country_SSA_HE2.shp"
).resolve()

output_png = Path(
    r"figures/WBGT_extreme_heat_exposure_days_combined.png"
).resolve()

output_png.parent.mkdir(parents=True, exist_ok=True)

FIELD = "Heat_expos"
LEGEND_TITLE = "Extreme heat exposure\n(WBGT$_{max}$ > 30℃)"

# =====================
# 2. Style
# =====================
mpl.rcParams.update({
    "font.size": 8,
    "axes.linewidth": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

LINE_COLORS = {
    "mean": "#3F8FA3",
    "p90": "#9A8CC8",
    "p95": "#D07868",
    "trend": "#333333",
    "any": "#6B9BBC",
    "moderate": "#A999D3",
    "severe": "#D1847A",
    "grid": "#E0E5E7",
    "text": "#2F2F2F",
}
MAP_COLORS = ["#F8F1EA", "#F0D9CA", "#E7B899", "#D88762", "#B9563A"]
EDGE_COLOR = "#FFFFFF"
COUNTRY_EDGE_COLOR = "#4A4A4A"


# =====================
# 3. Annual line-chart statistics
# =====================
records = []
tif_files = sorted(folder.glob("*.tif"))
if not tif_files:
    raise FileNotFoundError(f"No .tif files found in {folder}")

for tif_path in tif_files:
    year_match = re.search(r"(20\d{2})", tif_path.name)
    if year_match is None:
        print(f"Skip file without year: {tif_path.name}")
        continue

    year = int(year_match.group(1))
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        valid_mask = np.isfinite(data)
        if src.nodata is not None:
            valid_mask &= data != src.nodata
        valid_mask &= data >= 0
        valid_values = data[valid_mask]

    if valid_values.size == 0:
        print(f"No valid pixels in: {tif_path.name}")
        continue

    records.append({
        "year": year,
        "mean": float(np.mean(valid_values)),
        "p90": float(np.percentile(valid_values, 90)),
        "p95": float(np.percentile(valid_values, 95)),
        "share_any": float(np.mean(valid_values > 0) * 100),
        "share_15": float(np.mean(valid_values >= 15) * 100),
        "share_30": float(np.mean(valid_values >= 30) * 100),
        "n_pixels": int(valid_values.size),
    })

records = sorted(records, key=lambda x: x["year"])
if len(records) < 2:
    raise ValueError("At least two annual rasters are needed to draw a trend figure.")

years = np.array([r["year"] for r in records])
mean_values = np.array([r["mean"] for r in records])
p90_values = np.array([r["p90"] for r in records])
p95_values = np.array([r["p95"] for r in records])
share_any = np.array([r["share_any"] for r in records])
share_15 = np.array([r["share_15"] for r in records])
share_30 = np.array([r["share_30"] for r in records])


# =====================
# 4. Map data and classes
# =====================
gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)

if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    country_gdf = country_gdf.to_crs(epsg=4326)

valid = gdf.loc[gdf[FIELD].notna()].copy()
positive = valid.loc[valid[FIELD] > 0, FIELD]
if positive.empty:
    raise ValueError(f"{FIELD} has no positive values.")

eps = positive.min() * 0.999
bounds = np.array([
    0,
    eps,
    positive.quantile(0.25),
    positive.quantile(0.50),
    positive.quantile(0.75),
    positive.quantile(1.00),
], dtype=float)
bounds = np.unique(bounds)
n_classes = len(bounds) - 1
if n_classes < 2:
    raise ValueError("Not enough unique values to build discrete map classes.")

if n_classes <= len(MAP_COLORS):
    colors = MAP_COLORS[:n_classes]
else:
    base_cmap = plt.get_cmap("OrRd", n_classes + 2)
    colors = [mpl.colors.to_hex(base_cmap(i + 1)) for i in range(n_classes)]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, cmap.N)
zero_nodata_color = colors[0]


def format_heat_days_value(x):
    if x == 0:
        return "0"
    if x < 0.01:
        return ">0"
    if x < 1:
        return f"{x:.2f}"
    if x < 10:
        return f"{x:.1f}"
    return f"{x:,.0f}"


def add_scalebar(ax, length_km=1000, location=(0.29, 0.15), linewidth=1.35):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)
    km_per_deg_lon = 111.32 * math.cos(math.radians(y0))
    if km_per_deg_lon <= 0:
        km_per_deg_lon = 111.32

    length_deg = length_km / km_per_deg_lon
    x1 = x0 + length_deg
    right_margin = xmax - 0.03 * (xmax - xmin)
    if x1 > right_margin:
        x1 = right_margin
        x0 = x1 - length_deg

    ax.plot(
        [x0, x1], [y0, y0],
        color="#333333",
        linewidth=linewidth,
        solid_capstyle="butt",
        zorder=20,
    )
    ax.text(
        (x0 + x1) / 2, y0 + 0.9,
        f"{length_km:,} km",
        ha="center",
        va="bottom",
        fontsize=7.5,
        color="#333333",
        zorder=21,
    )
    return ((x0 + x1) / 2 - xmin) / (xmax - xmin), (y0 - ymin) / (ymax - ymin)


def add_north_arrow(ax, location=(0.40, 0.25), arrow_length=0.033):
    x, y = location
    ax.annotate(
        "",
        xy=(x, y + arrow_length),
        xytext=(x, y),
        xycoords=ax.transAxes,
        arrowprops=dict(
            arrowstyle="-|>",
            lw=1.0,
            color="#333333",
            mutation_scale=10,
        ),
        zorder=30,
    )
    ax.text(
        x,
        y + arrow_length + 0.008,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=8.5,
        fontweight="bold",
        color="#333333",
        zorder=31,
    )


# =====================
# 5. Composite figure
# =====================
fig = plt.figure(figsize=(12.4, 6.9), dpi=300, facecolor="white")
gs = GridSpec(
    2,
    2,
    figure=fig,
    width_ratios=[2.00, 0.82],
    height_ratios=[1, 1],
    wspace=0.03,
    hspace=0.28,
)
ax_map = fig.add_subplot(gs[:, 0])
ax0 = fig.add_subplot(gs[0, 1])
ax1 = fig.add_subplot(gs[1, 1], sharex=ax0)


ax_map.set_facecolor("white")
gdf.boundary.plot(
    ax=ax_map,
    linewidth=0.14,
    edgecolor=EDGE_COLOR,
    alpha=0.55,
    zorder=1,
)
gdf.plot(
    column=FIELD,
    ax=ax_map,
    cmap=cmap,
    norm=norm,
    linewidth=0.16,
    edgecolor=EDGE_COLOR,
    legend=False,
    missing_kwds={"color": zero_nodata_color, "edgecolor": EDGE_COLOR},
    zorder=2,
)
country_gdf.boundary.plot(
    ax=ax_map,
    edgecolor=COUNTRY_EDGE_COLOR,
    linewidth=0.34,
    alpha=0.85,
    facecolor="none",
    zorder=3,
)
country_gdf.dissolve().boundary.plot(
    ax=ax_map,
    edgecolor="#303030",
    linewidth=0.58,
    facecolor="none",
    zorder=4,
)

xmin, ymin, xmax, ymax = gdf.total_bounds
xpad = (xmax - xmin) * 0.025
ypad = (ymax - ymin) * 0.025
ax_map.set_xlim(xmin - xpad, xmax + xpad)
ax_map.set_ylim(ymin - ypad, ymax + ypad)
mean_lat = (ymin + ymax) / 2
ax_map.set_aspect(1 / np.cos(np.deg2rad(mean_lat)))
ax_map.set_xticks([])
ax_map.set_yticks([])
ax_map.tick_params(
    left=False,
    right=False,
    bottom=False,
    top=False,
    labelleft=False,
    labelbottom=False,
)
for spine in ax_map.spines.values():
    spine.set_visible(False)

legend_handles = []
for i in range(n_classes):
    lower = bounds[i]
    upper = bounds[i + 1]
    if i == 0:
        label = "0"
    elif i == 1:
        label = f">0-{format_heat_days_value(upper)}"
    else:
        label = f"{format_heat_days_value(lower)}-{format_heat_days_value(upper)}"
    legend_handles.append(Patch(facecolor=colors[i], edgecolor="none", label=label))

legend = ax_map.legend(
    handles=legend_handles,
    title=LEGEND_TITLE,
    loc="lower left",
    bbox_to_anchor=(0.015, 0.13),
    frameon=False,
    fontsize=7.2,
    title_fontsize=8.1,
    handlelength=1.35,
    handleheight=0.8,
    handletextpad=0.7,
    labelspacing=0.48,
    borderpad=0,
)
legend.get_title().set_fontweight("semibold")
legend.get_title().set_color("#333333")

scalebar_center_x, scalebar_y = add_scalebar(
    ax_map,
    length_km=1000,
    location=(0.29, 0.15),
    linewidth=1.35,
)
add_north_arrow(
    ax_map,
    location=(scalebar_center_x, scalebar_y + 0.075),
    arrow_length=0.033,
)
ax_map.text(
    0.015,
    0.985,
    "(a)",
    transform=ax_map.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    fontweight="bold",
    color="#222222",
)


def style_bar_axis(ax):
    ax.grid(axis="y", color=LINE_COLORS["grid"], linewidth=0.55, alpha=0.85)
    ax.tick_params(axis="both", length=3, width=0.65, color="#333333", labelsize=7.2)
    ax.spines["left"].set_color("#333333")
    ax.spines["bottom"].set_color("#333333")
    ax.spines["left"].set_linewidth(0.65)
    ax.spines["bottom"].set_linewidth(0.65)
    ax.set_axisbelow(True)


heat_metric_label = (
    "HI$_{max}$ > 40.6℃"
    if "HI406" in str(folder)
    else "WBGT$_{max}$ > 30℃"
)
x = np.arange(len(years))
bar_width = 0.23
bar_edge = "#FFFFFF"
bar_alpha = 0.62

trend_coef = np.polyfit(years, mean_values, deg=1)
trend_values = np.polyval(trend_coef, years)
slope = trend_coef[0]
total_change = trend_values[-1] - trend_values[0]

ax0.bar(
    x - bar_width,
    mean_values,
    width=bar_width,
    color=LINE_COLORS["mean"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label="Mean",
    zorder=3,
)
ax0.bar(
    x,
    p90_values,
    width=bar_width,
    color=LINE_COLORS["p90"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label="P90",
    zorder=3,
)
ax0.bar(
    x + bar_width,
    p95_values,
    width=bar_width,
    color=LINE_COLORS["p95"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label="P95",
    zorder=3,
)
ax0.plot(
    x - bar_width,
    mean_values,
    color=LINE_COLORS["mean"],
    marker="o",
    linewidth=1.05,
    markersize=2.7,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax0.plot(
    x,
    p90_values,
    color=LINE_COLORS["p90"],
    marker="s",
    linewidth=1.05,
    markersize=2.6,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax0.plot(
    x + bar_width,
    p95_values,
    color=LINE_COLORS["p95"],
    marker="^",
    linewidth=1.05,
    markersize=2.8,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)

ax0.text(
    0.03,
    0.95,
    f"Mean fitted change: {total_change:+.1f} days\n{slope:+.2f} days yr$^{{-1}}$",
    transform=ax0.transAxes,
    ha="left",
    va="top",
    fontsize=6.2,
    color=LINE_COLORS["text"],
    bbox=dict(
        boxstyle="round,pad=0.22",
        facecolor="white",
        edgecolor="#D0D4D8",
        linewidth=0.55,
    ),
)
ax0.set_title(
    "(b)  Intensity and upper tail",
    loc="left",
    fontsize=8.4,
    pad=5,
    fontweight="semibold",
)
ax0.set_ylabel(f"Days per pixel\n({heat_metric_label})", fontsize=7.5)
ax0.set_xlim(-0.6, len(years) - 0.4)
ax0.set_ylim(bottom=0)
ax0.set_xticks(x)
ax0.set_xticklabels([])
ax0.legend(
    loc="upper right",
    ncol=3,
    fontsize=6.7,
    handlelength=1.2,
    columnspacing=0.8,
    borderpad=0.1,
)
style_bar_axis(ax0)

ax1.bar(
    x - bar_width,
    share_any,
    width=bar_width,
    color=LINE_COLORS["any"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label=">0 days",
    zorder=3,
)
ax1.bar(
    x,
    share_15,
    width=bar_width,
    color=LINE_COLORS["moderate"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label=">=15 days",
    zorder=3,
)
ax1.bar(
    x + bar_width,
    share_30,
    width=bar_width,
    color=LINE_COLORS["severe"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label=">=30 days",
    zorder=3,
)
ax1.plot(
    x - bar_width,
    share_any,
    color=LINE_COLORS["any"],
    marker="o",
    linewidth=1.05,
    markersize=2.7,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax1.plot(
    x,
    share_15,
    color=LINE_COLORS["moderate"],
    marker="s",
    linewidth=1.05,
    markersize=2.6,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax1.plot(
    x + bar_width,
    share_30,
    color=LINE_COLORS["severe"],
    marker="^",
    linewidth=1.05,
    markersize=2.8,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)

ax1.set_title(
    "(c)  Spatial extent",
    loc="left",
    fontsize=8.4,
    pad=5,
    fontweight="semibold",
)
ax1.set_ylabel("Valid pixels exposed (%)", fontsize=7.5)
ax1.set_xlabel("Year", fontsize=7.5)
ax1.set_xticks(x)
ax1.set_xticklabels(years)
ax1.set_xlim(-0.6, len(years) - 0.4)
ax1.set_ylim(bottom=0)
ax1.legend(
    loc="lower right",
    bbox_to_anchor=(1.0, 1.02),
    ncol=3,
    fontsize=6.6,
    handlelength=1.0,
    columnspacing=0.8,
    handletextpad=0.35,
    borderaxespad=0.0,
    borderpad=0.1,
)
style_bar_axis(ax1)

fig.subplots_adjust(
    left=0.02,
    right=0.985,
    bottom=0.085,
    top=0.965,
)

try:
    fig.savefig(
        output_png,
        dpi=600,
        facecolor="white",
    )
except PermissionError:
    output_png = output_png.with_name(output_png.stem + "_new.png")
    fig.savefig(
        output_png,
        dpi=600,
        facecolor="white",
    )

plt.show()

# HI extreme heat exposure and days

In [ ]:
from pathlib import Path
import re
import math

import numpy as np
import geopandas as gpd
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch


# =====================
# 1. Paths
# =====================

folder = Path(r"../data/HI406_days").resolve()
shp_path = Path(r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp").resolve()
country_shp_path = Path(r"../data/country/country_SSA_HE2.shp").resolve()
output_png = Path(r"figures/HI406_extreme_heat_exposure_days_combined.png").resolve()
output_png.parent.mkdir(parents=True, exist_ok=True)

FIELD = "HI_expo"
LEGEND_TITLE = "Extreme heat exposure\n(HI$_{max}$ > 40.6℃)"


# =====================
# 2. Style
# =====================
mpl.rcParams.update({
    "font.size": 8,
    "axes.linewidth": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

LINE_COLORS = {
    "mean": "#3F8FA3",
    "p90": "#9A8CC8",
    "p95": "#D07868",
    "trend": "#333333",
    "any": "#6B9BBC",
    "moderate": "#A999D3",
    "severe": "#D1847A",
    "grid": "#E0E5E7",
    "text": "#2F2F2F",
}
MAP_COLORS = ["#F8F1EA", "#F0D9CA", "#E7B899", "#D88762", "#B9563A"]
EDGE_COLOR = "#FFFFFF"
COUNTRY_EDGE_COLOR = "#4A4A4A"


# =====================
# 3. Annual line-chart statistics
# =====================
records = []
tif_files = sorted(folder.glob("*.tif"))
if not tif_files:
    raise FileNotFoundError(f"No .tif files found in {folder}")

for tif_path in tif_files:
    year_match = re.search(r"(20\d{2})", tif_path.name)
    if year_match is None:
        print(f"Skip file without year: {tif_path.name}")
        continue

    year = int(year_match.group(1))
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        valid_mask = np.isfinite(data)
        if src.nodata is not None:
            valid_mask &= data != src.nodata
        valid_mask &= data >= 0
        valid_values = data[valid_mask]

    if valid_values.size == 0:
        print(f"No valid pixels in: {tif_path.name}")
        continue

    records.append({
        "year": year,
        "mean": float(np.mean(valid_values)),
        "p90": float(np.percentile(valid_values, 90)),
        "p95": float(np.percentile(valid_values, 95)),
        "share_any": float(np.mean(valid_values > 0) * 100),
        "share_15": float(np.mean(valid_values >= 15) * 100),
        "share_30": float(np.mean(valid_values >= 30) * 100),
        "n_pixels": int(valid_values.size),
    })

records = sorted(records, key=lambda x: x["year"])
if len(records) < 2:
    raise ValueError("At least two annual rasters are needed to draw a trend figure.")

years = np.array([r["year"] for r in records])
mean_values = np.array([r["mean"] for r in records])
p90_values = np.array([r["p90"] for r in records])
p95_values = np.array([r["p95"] for r in records])
share_any = np.array([r["share_any"] for r in records])
share_15 = np.array([r["share_15"] for r in records])
share_30 = np.array([r["share_30"] for r in records])


# =====================
# 4. Map data and classes
# =====================
gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)

if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    country_gdf = country_gdf.to_crs(epsg=4326)

valid = gdf.loc[gdf[FIELD].notna()].copy()
positive = valid.loc[valid[FIELD] > 0, FIELD]
if positive.empty:
    raise ValueError(f"{FIELD} has no positive values.")

eps = positive.min() * 0.999
bounds = np.array([
    0,
    eps,
    positive.quantile(0.25),
    positive.quantile(0.50),
    positive.quantile(0.75),
    positive.quantile(1.00),
], dtype=float)
bounds = np.unique(bounds)
n_classes = len(bounds) - 1
if n_classes < 2:
    raise ValueError("Not enough unique values to build discrete map classes.")

if n_classes <= len(MAP_COLORS):
    colors = MAP_COLORS[:n_classes]
else:
    base_cmap = plt.get_cmap("OrRd", n_classes + 2)
    colors = [mpl.colors.to_hex(base_cmap(i + 1)) for i in range(n_classes)]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, cmap.N)
zero_nodata_color = colors[0]


def format_heat_days_value(x):
    if x == 0:
        return "0"
    if x < 0.01:
        return ">0"
    if x < 1:
        return f"{x:.2f}"
    if x < 10:
        return f"{x:.1f}"
    return f"{x:,.0f}"
def add_scalebar(ax, length_km=1000, location=(0.29, 0.15), linewidth=1.35):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)
    km_per_deg_lon = 111.32 * math.cos(math.radians(y0))
    if km_per_deg_lon <= 0:
        km_per_deg_lon = 111.32

    length_deg = length_km / km_per_deg_lon
    x1 = x0 + length_deg
    right_margin = xmax - 0.03 * (xmax - xmin)
    if x1 > right_margin:
        x1 = right_margin
        x0 = x1 - length_deg

    ax.plot(
        [x0, x1], [y0, y0],
        color="#333333",
        linewidth=linewidth,
        solid_capstyle="butt",
        zorder=20,
    )
    ax.text(
        (x0 + x1) / 2, y0 + 0.9,
        f"{length_km:,} km",
        ha="center",
        va="bottom",
        fontsize=7.5,
        color="#333333",
        zorder=21,
    )
    return ((x0 + x1) / 2 - xmin) / (xmax - xmin), (y0 - ymin) / (ymax - ymin)


def add_north_arrow(ax, location=(0.40, 0.25), arrow_length=0.033):
    x, y = location
    ax.annotate(
        "",
        xy=(x, y + arrow_length),
        xytext=(x, y),
        xycoords=ax.transAxes,
        arrowprops=dict(
            arrowstyle="-|>",
            lw=1.0,
            color="#333333",
            mutation_scale=10,
        ),
        zorder=30,
    )
    ax.text(
        x,
        y + arrow_length + 0.008,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=8.5,
        fontweight="bold",
        color="#333333",
        zorder=31,
    )


# =====================
# 5. Composite figure
# =====================
fig = plt.figure(figsize=(12.4, 6.9), dpi=300, facecolor="white")
gs = GridSpec(
    2,
    2,
    figure=fig,
    width_ratios=[2.00, 0.82],
    height_ratios=[1, 1],
    wspace=0.03,
    hspace=0.28,
)
ax_map = fig.add_subplot(gs[:, 0])
ax0 = fig.add_subplot(gs[0, 1])
ax1 = fig.add_subplot(gs[1, 1], sharex=ax0)


ax_map.set_facecolor("white")
gdf.boundary.plot(
    ax=ax_map,
    linewidth=0.14,
    edgecolor=EDGE_COLOR,
    alpha=0.55,
    zorder=1,
)
gdf.plot(
    column=FIELD,
    ax=ax_map,
    cmap=cmap,
    norm=norm,
    linewidth=0.16,
    edgecolor=EDGE_COLOR,
    legend=False,
    missing_kwds={"color": zero_nodata_color, "edgecolor": EDGE_COLOR},
    zorder=2,
)
country_gdf.boundary.plot(
    ax=ax_map,
    edgecolor=COUNTRY_EDGE_COLOR,
    linewidth=0.34,
    alpha=0.85,
    facecolor="none",
    zorder=3,
)
country_gdf.dissolve().boundary.plot(
    ax=ax_map,
    edgecolor="#303030",
    linewidth=0.58,
    facecolor="none",
    zorder=4,
)

xmin, ymin, xmax, ymax = gdf.total_bounds
xpad = (xmax - xmin) * 0.025
ypad = (ymax - ymin) * 0.025
ax_map.set_xlim(xmin - xpad, xmax + xpad)
ax_map.set_ylim(ymin - ypad, ymax + ypad)
mean_lat = (ymin + ymax) / 2
ax_map.set_aspect(1 / np.cos(np.deg2rad(mean_lat)))
ax_map.set_xticks([])
ax_map.set_yticks([])
ax_map.tick_params(
    left=False,
    right=False,
    bottom=False,
    top=False,
    labelleft=False,
    labelbottom=False,
)
for spine in ax_map.spines.values():
    spine.set_visible(False)

legend_handles = []
for i in range(n_classes):
    lower = bounds[i]
    upper = bounds[i + 1]
    if i == 0:
        label = "0"
    elif i == 1:
        label = f">0-{format_heat_days_value(upper)}"
    else:
        label = f"{format_heat_days_value(lower)}-{format_heat_days_value(upper)}"
    legend_handles.append(Patch(facecolor=colors[i], edgecolor="none", label=label))

legend = ax_map.legend(
    handles=legend_handles,
    title=LEGEND_TITLE,
    loc="lower left",
    bbox_to_anchor=(0.015, 0.13),
    frameon=False,
    fontsize=7.2,
    title_fontsize=8.1,
    handlelength=1.35,
    handleheight=0.8,
    handletextpad=0.7,
    labelspacing=0.48,
    borderpad=0,
)
legend.get_title().set_fontweight("semibold")
legend.get_title().set_color("#333333")

scalebar_center_x, scalebar_y = add_scalebar(
    ax_map,
    length_km=1000,
    location=(0.29, 0.15),
    linewidth=1.35,
)
add_north_arrow(
    ax_map,
    location=(scalebar_center_x, scalebar_y + 0.075),
    arrow_length=0.033,
)
ax_map.text(
    0.015,
    0.985,
    "(a)",
    transform=ax_map.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    fontweight="bold",
    color="#222222",
)


def style_bar_axis(ax):
    ax.grid(axis="y", color=LINE_COLORS["grid"], linewidth=0.55, alpha=0.85)
    ax.tick_params(axis="both", length=3, width=0.65, color="#333333", labelsize=7.2)
    ax.spines["left"].set_color("#333333")
    ax.spines["bottom"].set_color("#333333")
    ax.spines["left"].set_linewidth(0.65)
    ax.spines["bottom"].set_linewidth(0.65)
    ax.set_axisbelow(True)


heat_metric_label = (
    "HI$_{max}$ > 40.6℃"
    if "HI406" in str(folder)
    else "WBGT$_{max}$ > 30℃"
)
x = np.arange(len(years))
bar_width = 0.23
bar_edge = "#FFFFFF"
bar_alpha = 0.62

trend_coef = np.polyfit(years, mean_values, deg=1)
trend_values = np.polyval(trend_coef, years)
slope = trend_coef[0]
total_change = trend_values[-1] - trend_values[0]

ax0.bar(
    x - bar_width,
    mean_values,
    width=bar_width,
    color=LINE_COLORS["mean"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label="Mean",
    zorder=3,
)
ax0.bar(
    x,
    p90_values,
    width=bar_width,
    color=LINE_COLORS["p90"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label="P90",
    zorder=3,
)
ax0.bar(
    x + bar_width,
    p95_values,
    width=bar_width,
    color=LINE_COLORS["p95"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label="P95",
    zorder=3,
)
ax0.plot(
    x - bar_width,
    mean_values,
    color=LINE_COLORS["mean"],
    marker="o",
    linewidth=1.05,
    markersize=2.7,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax0.plot(
    x,
    p90_values,
    color=LINE_COLORS["p90"],
    marker="s",
    linewidth=1.05,
    markersize=2.6,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax0.plot(
    x + bar_width,
    p95_values,
    color=LINE_COLORS["p95"],
    marker="^",
    linewidth=1.05,
    markersize=2.8,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)

ax0.text(
    0.03,
    0.95,
    f"Mean fitted change: {total_change:+.1f} days\n{slope:+.2f} days yr$^{{-1}}$",
    transform=ax0.transAxes,
    ha="left",
    va="top",
    fontsize=6.2,
    color=LINE_COLORS["text"],
    bbox=dict(
        boxstyle="round,pad=0.22",
        facecolor="white",
        edgecolor="#D0D4D8",
        linewidth=0.55,
    ),
)
ax0.set_title(
    "(b)  Intensity and upper tail",
    loc="left",
    fontsize=8.4,
    pad=5,
    fontweight="semibold",
)
ax0.set_ylabel(f"Days per pixel\n({heat_metric_label})", fontsize=7.5)
ax0.set_xlim(-0.6, len(years) - 0.4)
ax0.set_ylim(bottom=0)
ax0.set_xticks(x)
ax0.set_xticklabels([])
ax0.legend(
    loc="upper right",
    ncol=3,
    fontsize=6.7,
    handlelength=1.2,
    columnspacing=0.8,
    borderpad=0.1,
)
style_bar_axis(ax0)

ax1.bar(
    x - bar_width,
    share_any,
    width=bar_width,
    color=LINE_COLORS["any"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label=">0 days",
    zorder=3,
)
ax1.bar(
    x,
    share_15,
    width=bar_width,
    color=LINE_COLORS["moderate"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label=">=15 days",
    zorder=3,
)
ax1.bar(
    x + bar_width,
    share_30,
    width=bar_width,
    color=LINE_COLORS["severe"],
    edgecolor=bar_edge,
    linewidth=0.45,
    alpha=bar_alpha,
    label=">=30 days",
    zorder=3,
)
ax1.plot(
    x - bar_width,
    share_any,
    color=LINE_COLORS["any"],
    marker="o",
    linewidth=1.05,
    markersize=2.7,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax1.plot(
    x,
    share_15,
    color=LINE_COLORS["moderate"],
    marker="s",
    linewidth=1.05,
    markersize=2.6,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)
ax1.plot(
    x + bar_width,
    share_30,
    color=LINE_COLORS["severe"],
    marker="^",
    linewidth=1.05,
    markersize=2.8,
    markeredgecolor="white",
    markeredgewidth=0.45,
    label="_nolegend_",
    zorder=4,
)

ax1.set_title(
    "(c)  Spatial extent",
    loc="left",
    fontsize=8.4,
    pad=5,
    fontweight="semibold",
)
ax1.set_ylabel("Valid pixels exposed (%)", fontsize=7.5)
ax1.set_xlabel("Year", fontsize=7.5)
ax1.set_xticks(x)
ax1.set_xticklabels(years)
ax1.set_xlim(-0.6, len(years) - 0.4)
ax1.set_ylim(bottom=0)
ax1.legend(
    loc="lower right",
    bbox_to_anchor=(1.0, 1.02),
    ncol=3,
    fontsize=6.6,
    handlelength=1.0,
    columnspacing=0.8,
    handletextpad=0.35,
    borderaxespad=0.0,
    borderpad=0.1,
)
style_bar_axis(ax1)

fig.subplots_adjust(
    left=0.02,
    right=0.985,
    bottom=0.085,
    top=0.965,
)

try:
    fig.savefig(
        output_png,
        dpi=600,
        facecolor="white",
    )
except PermissionError:
    output_png = output_png.with_name(output_png.stem + "_new.png")
    fig.savefig(
        output_png,
        dpi=600,
        facecolor="white",
    )

plt.show()

# flood

In [ ]:
from pathlib import Path

import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

shp_path = Path(
    r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
).resolve()

country_shp_path = Path(
    r"../data/country/country_SSA_HE2.shp"
).resolve()

output_png = Path(
    r"figures/flood_metrics.png"
).resolve()

output_png.parent.mkdir(parents=True, exist_ok=True)

AREA_FIELDS = {
    "Low": "fl_lowkm2",
    "Medium": "fl_medkm2",
    "High": "fl_hikm2",
}

MAP_SPECS = [
    {
        "field": "fl_wsev",
        "title": "(b) Area-weighted flood severity",
        "legend_title": "Severity score",
        "fmt": "score",
    },
    {
        "field": "sl_sevsum",
        "title": "(d) Slum flood severity burden",
        "legend_title": "People x severity",
        "fmt": "people",
    },
]

BLUE_MAP = ["#F3F8FC", "#D8ECF7", "#AFD8EC", "#72B7D8", "#2F8FBC", "#126A9A"]
CHART_BLUE = "#287EA8"
DARK_BLUE = "#0F4F7A"
MID_BLUE = "#5FAED1"
LIGHT_BLUE = "#BBDDED"
TEXT = "#26343C"
MUTED = "#667882"
BORDER = "#FFFFFF"
NA_COLOR = "#F1F3F4"
OCEAN = "#FFFFFF"

mpl.rcParams.update({
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.4,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#D3DCE2",
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "savefig.facecolor": "white",
})


def format_compact(value, fmt="people"):
    if value is None or not np.isfinite(value):
        return "NA"
    value = float(value)
    sign = "-" if value < 0 else ""
    value = abs(value)
    if fmt == "score":
        return f"{sign}{value:.1f}"
    if value >= 1_000_000_000:
        return f"{sign}{value / 1_000_000_000:.1f}B"
    if value >= 1_000_000:
        return f"{sign}{value / 1_000_000:.1f}M"
    if value >= 1_000:
        return f"{sign}{value / 1_000:.0f}K"
    return f"{sign}{value:.0f}"


def make_zero_positive_classes(values, n_classes=5):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.array([0, 1]), False
    vals = vals[vals >= 0]
    positive = vals[vals > 0]
    if positive.size == 0:
        return np.array([0, 1]), False
    quantiles = np.linspace(0, 1, n_classes + 1)[1:]
    cuts = np.quantile(positive, quantiles)
    edges = np.unique(np.r_[0, cuts])
    if edges.size < 2:
        edges = np.array([0, positive.max()])
    if edges[-1] <= edges[0]:
        edges[-1] = edges[0] + 1
    return edges, True


def legend_handles(edges, colors, fmt):
    handles = []
    if len(edges) >= 2:
        handles.append(Patch(facecolor=colors[0], edgecolor="none", label="0"))
    for idx in range(1, len(edges)):
        low = edges[idx - 1]
        high = edges[idx]
        if idx == 1:
            label = f">0 to {format_compact(high, fmt)}"
        else:
            label = f"{format_compact(low, fmt)} to {format_compact(high, fmt)}"
        handles.append(Patch(facecolor=colors[min(idx, len(colors)-1)], edgecolor="none", label=label))
    return handles


def setup_map_axis(ax, title):
    ax.set_facecolor(OCEAN)
    ax.set_title(title, loc="left", pad=5, color=TEXT, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_discrete_map(ax, gdf, field, title, legend_title, fmt):
    setup_map_axis(ax, title)
    values = gdf[field].replace([np.inf, -np.inf], np.nan)
    edges, has_positive = make_zero_positive_classes(values)
    colors = BLUE_MAP[: max(len(edges), 2)]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(edges, cmap.N, clip=True)
    gdf.plot(ax=ax, color=NA_COLOR, edgecolor=BORDER, linewidth=0.10, zorder=1)
    if has_positive:
        gdf.assign(_plot_value=values.fillna(-1)).plot(
            column="_plot_value",
            ax=ax,
            cmap=cmap,
            norm=norm,
            edgecolor=BORDER,
            linewidth=0.10,
            missing_kwds={"color": NA_COLOR},
            zorder=2,
        )
        handles = legend_handles(edges, colors, fmt)
    else:
        handles = [Patch(facecolor=colors[0], edgecolor="none", label="0")]

    country_gdf.boundary.plot(
        ax=ax,
        edgecolor="#222222",
        linewidth=0.26,
        alpha=0.85,
        facecolor="none",
        zorder=3,
    )
    country_outer_boundary.boundary.plot(
        ax=ax,
        edgecolor="#111111",
        linewidth=0.50,
        facecolor="none",
        zorder=4,
    )

    leg = ax.legend(
        handles=handles,
        title=legend_title,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.02),
        frameon=True,
        framealpha=0.94,
        borderpad=0.45,
        labelspacing=0.36,
        handlelength=0.9,
        handleheight=0.55,
    )
    leg.get_title().set_fontsize(7.6)
    leg.get_frame().set_edgecolor("#CAD7DE")
    leg.get_frame().set_linewidth(0.5)
    ax.set_aspect("equal")


def style_chart_axis(ax, grid_axis="x"):
    ax.grid(axis=grid_axis, color="#E0E0E0", linewidth=0.6, alpha=0.85)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#000000")
    ax.spines["bottom"].set_color("#000000")
    ax.tick_params(axis="both", colors="#000000")
    ax.xaxis.label.set_color("#000000")
    ax.yaxis.label.set_color("#000000")


required_fields = list(AREA_FIELDS.values()) + [
    "fl_wsev", "sl_sevsum", "sl_pop", "sl_flpop", "sl_flwsev"
]
gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)
if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)
country_outer_boundary = country_gdf.dissolve()
missing = [field for field in required_fields if field not in gdf.columns]
if missing:
    raise KeyError(f"Missing required flood fields: {missing}")

for field in required_fields:
    gdf[field] = gdf[field].replace([np.inf, -np.inf], np.nan)

fig, axes = plt.subplots(
    2, 2,
    figsize=(12.2, 8.6),
    dpi=300,
    gridspec_kw={"width_ratios": [0.82, 1.18], "wspace": 0.07, "hspace": 0.18},
)
ax_a, ax_b, ax_c, ax_d = axes.ravel()

# Panel a: flooded area by severity.
severity_labels = list(AREA_FIELDS.keys())
area_km2 = np.array([gdf[field].fillna(0).sum() for field in AREA_FIELDS.values()], dtype=float)
area_million = area_km2 / 1_000_000

severity_colors = ["#BBDDED", "#5FAED1", "#287EA8"]
y = np.arange(len(severity_labels))
ax_a.barh(y, area_million, height=0.38, color=severity_colors, edgecolor="white", linewidth=0.7)
label_offset = max(area_million.max(), 1) * 0.018
for yi, val in zip(y, area_million):
    ax_a.text(val + label_offset, yi, f"{val:.2f}", va="center", ha="left", color= "Black", fontsize=7.4)
ax_a.set_yticks(y)
ax_a.set_yticklabels(severity_labels)
ax_a.invert_yaxis()
ax_a.set_xlabel("Flooded area (million km2)")
ax_a.set_title("(a) Flooded area by severity", loc="left", pad=5, color=TEXT, fontweight="bold")
style_chart_axis(ax_a, "x")
ax_a.set_xlim(0, area_million.max() * 1.18)

# Panel b: map of area-weighted flood severity.
plot_discrete_map(
    ax_b, gdf,
    field="fl_wsev",
    title="(b) Area-weighted flood severity",
    legend_title="Severity score",
    fmt="score",
)

# Panel c: relationship between total slum population and flood-exposed slum population.
scatter_df = gdf[["sl_pop", "sl_flpop"]].dropna().copy()
scatter_df = scatter_df[(scatter_df["sl_pop"] > 0) & (scatter_df["sl_flpop"] > 0)]
ax_c.scatter(
    scatter_df["sl_pop"],
    scatter_df["sl_flpop"],
    s=9,
    color=CHART_BLUE,
    alpha=0.38,
    linewidths=0,
    rasterized=True,
)
if not scatter_df.empty:
    lim_min = max(1, min(scatter_df["sl_pop"].min(), scatter_df["sl_flpop"].min()) * 0.75)
    lim_max = max(scatter_df["sl_pop"].max(), scatter_df["sl_flpop"].max()) * 1.35
    ax_c.plot([lim_min, lim_max], [lim_min, lim_max], color=DARK_BLUE, linewidth=0.9, linestyle="--", label="1:1")
    ax_c.set_xlim(lim_min, lim_max)
    ax_c.set_ylim(lim_min, lim_max)
ax_c.set_xscale("log")
ax_c.set_yscale("log")
ax_c.set_xlabel("Slum population")
ax_c.set_ylabel("Flood-exposed slum population")
ax_c.set_title("(c) Slum population exposed to flooding", loc="left", pad=5, color=TEXT, fontweight="bold")
ax_c.legend(loc="lower right", frameon=False, handlelength=1.6)
style_chart_axis(ax_c, "both")

# Panel d: map of slum flood severity burden.
plot_discrete_map(
    ax_d, gdf,
    field="sl_sevsum",
    title="(d) Slum flood severity burden",
    legend_title="Population x severity",
    fmt="people",
)

fig.savefig(output_png, dpi=600, bbox_inches="tight", pad_inches=0.08)
plt.show()


# EDI

In [ ]:
from pathlib import Path
import math

import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch


# =====================
# 1. Paths and fields
# =====================

shp_path = Path(r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp").resolve()
country_shp_path = Path(r"../data/country/country_SSA_HE2.shp").resolve()
output_png = Path(r"figures/EDI_maps.png").resolve()
output_png.parent.mkdir(parents=True, exist_ok=True)

MAP_SPECS = [
    {
        "field": "GDI",
        "title": "Greenness Deficit Index",
        "legend_title": "GDI",
        "colors": ["#EEF7EA", "#CFE8C9", "#9FD3B0", "#5DAE98", "#2C7F8E"],
        "fmt": "gdi",
    },
    {
        "field": "YongJiDu",
        "title": "Settlement Crowding",
        "legend_title": "YongJiDu",
        "colors": ["#F3EEF8", "#D8C7EC", "#B59ADA", "#8B6FC1", "#5E4A9A"],
        "fmt": "float2",
    },
    {
        "field": "Roof_Vul",
        "title": "Roof Vulnerability",
        "legend_title": "Roof_Vul",
        "colors": ["#FDF0F5", "#F6CBDC", "#E99AB8", "#D86593", "#B9336F"],
        "fmt": "small",
    },
    {
        "field": "EDI_qmean",
        "title": "Environmental Deprivation Index (EDI)",
        "legend_title": "EDI_qmean",
        "colors": ["#F7F0E8", "#E8D2BC", "#D0AA84", "#AA7A55", "#7C4B32"],
        "fmt": "float1",
    },
]


# =====================
# 2. Style
# =====================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8,
    "axes.linewidth": 0.7,
    "legend.frameon": False,
})

EDGE_COLOR = "#FFFFFF"
COUNTRY_EDGE_COLOR = "#4A4A4A"
OUTER_EDGE_COLOR = "#303030"
MISSING_COLOR = "#F3F1EC"
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)"]


def format_value(x, fmt):
    if not np.isfinite(x):
        return "NA"
    if fmt == "small":
        if abs(x) < 0.001:
            text = f"{x:.5f}"
        else:
            text = f"{x:.4f}"
    elif fmt == "gdi":
        text = f"{x:.2f}"
    elif fmt == "float1":
        text = f"{x:.1f}"
    elif fmt == "float2":
        text = f"{x:.2f}"
    elif abs(x) >= 1000:
        text = f"{x:,.0f}"
    else:
        text = f"{x:.2f}"
    return text


def make_quantile_classes(values, n_classes=5):
    valid_values = values[np.isfinite(values)]
    if valid_values.size == 0:
        raise ValueError("No valid values are available for map classification.")

    quantiles = np.linspace(0, 1, n_classes + 1)
    bounds = np.quantile(valid_values, quantiles)
    bounds = np.unique(bounds.astype(float))
    if len(bounds) < 3:
        vmin = float(np.min(valid_values))
        vmax = float(np.max(valid_values))
        if vmin == vmax:
            delta = abs(vmin) * 0.01 if vmin != 0 else 0.01
            bounds = np.array([vmin - delta, vmin + delta], dtype=float)
        else:
            bounds = np.linspace(vmin, vmax, min(n_classes, valid_values.size) + 1)
    return bounds


def build_legend_handles(bounds, colors, fmt):
    handles = []
    for i in range(len(bounds) - 1):
        lower = bounds[i]
        upper = bounds[i + 1]
        if i == 0:
            label = f"{format_value(lower, fmt)} to {format_value(upper, fmt)}"
        else:
            label = f">{format_value(lower, fmt)} to {format_value(upper, fmt)}"
        handles.append(Patch(facecolor=colors[i], edgecolor="none", label=label))
    return handles


# =====================
# 3. Read data
# =====================
gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)

required_fields = [spec["field"] for spec in MAP_SPECS]
missing_fields = [field for field in required_fields if field not in gdf.columns]
if missing_fields:
    raise KeyError(f"Missing required EDI fields: {missing_fields}")

if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    country_gdf = country_gdf.to_crs(epsg=4326)

MAIN_AFRICA_LON_MIN = -18
MAIN_AFRICA_LON_MAX = 52


def keep_main_africa(frame):
    points = frame.geometry.representative_point()
    mask = points.x.between(MAIN_AFRICA_LON_MIN, MAIN_AFRICA_LON_MAX)
    return frame.loc[mask].copy()


xmin, ymin, xmax, ymax = gdf.total_bounds
xpad = (xmax - xmin) * 0.025
ypad = (ymax - ymin) * 0.025
mean_lat = (ymin + ymax) / 2
aspect = 1 / np.cos(np.deg2rad(mean_lat))


# =====================
# 4. Plot
# =====================
fig, axes = plt.subplots(2, 2, figsize=(11.4, 8.6), dpi=300, facecolor="white")
axes = axes.ravel()

for ax, spec, panel_label in zip(axes, MAP_SPECS, PANEL_LABELS):
    field = spec["field"]
    panel_gdf = keep_main_africa(gdf) if field == "Roof_Vul" else gdf
    panel_country_gdf = keep_main_africa(country_gdf) if field == "Roof_Vul" else country_gdf
    panel_outer_boundary = panel_country_gdf.dissolve()
    values = gdf[field].astype(float).to_numpy()
    bounds = make_quantile_classes(values, n_classes=5)
    n_classes = len(bounds) - 1
    colors = spec["colors"][:n_classes]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(bounds, cmap.N)

    ax.set_facecolor("white")
    panel_gdf.boundary.plot(
        ax=ax,
        linewidth=0.10,
        edgecolor=EDGE_COLOR,
        alpha=0.55,
        zorder=1,
    )
    panel_gdf.plot(
        column=field,
        ax=ax,
        cmap=cmap,
        norm=norm,
        linewidth=0.11,
        edgecolor=EDGE_COLOR,
        legend=False,
        missing_kwds={"color": MISSING_COLOR, "edgecolor": EDGE_COLOR},
        zorder=2,
    )
    panel_country_gdf.boundary.plot(
        ax=ax,
        edgecolor=COUNTRY_EDGE_COLOR,
        linewidth=0.26,
        alpha=0.85,
        facecolor="none",
        zorder=3,
    )
    panel_outer_boundary.boundary.plot(
        ax=ax,
        edgecolor=OUTER_EDGE_COLOR,
        linewidth=0.50,
        facecolor="none",
        zorder=4,
    )

    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)
    ax.set_aspect(aspect)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(left=False, right=False, bottom=False, top=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(
        f"{panel_label}  {spec['title']}",
        loc="left",
        fontsize=8.8,
        pad=4,
        fontweight="semibold",
        color="#2F2F2F",
    )

    legend = ax.legend(
        handles=build_legend_handles(bounds, colors, spec["fmt"]),
        title=spec["title"],
        loc="lower left",
        bbox_to_anchor=(0.015, 0.055),
        frameon=False,
        fontsize=6.2,
        title_fontsize=6.8,
        handlelength=1.15,
        handleheight=0.65,
        handletextpad=0.55,
        labelspacing=0.34,
        borderpad=0,
    )
    legend.get_title().set_fontweight("semibold")
    legend.get_title().set_color("#333333")

fig.subplots_adjust(left=0.02, right=0.985, bottom=0.025, top=0.965, wspace=0.01, hspace=0.05)

try:
    fig.savefig(output_png, dpi=600, facecolor="white")
except PermissionError:
    output_png = output_png.with_name(output_png.stem + "_new.png")
    fig.savefig(output_png, dpi=600, facecolor="white")

plt.show()

# malaria map

In [ ]:
# Malaria burden maps
from pathlib import Path

import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

shp_path = Path(
    r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
).resolve()

country_shp_path = Path(
    r"../data/country/country_SSA_HE2.shp"
).resolve()

output_png = Path(
    r"figures/malaria_maps.png"
).resolve()
output_png.parent.mkdir(parents=True, exist_ok=True)

MAP_SPECS = [
    {
        "field": "sl_pfinf",
        "title": "(a) Slum malaria infection burden",
        "legend_title": "Infection burden",
        "fmt": "people",
        "colors": ["#FFF7EC", "#FEE8C8", "#FDD49E", "#FDBB84", "#F08A4B", "#C75B12"],
    },
    {
        "field": "sl_pfinc",
        "title": "(b) Slum malaria incidence burden",
        "legend_title": "Incidence burden",
        "fmt": "people",
        "colors": ["#FFF5F0", "#FEE0D2", "#FCBBA1", "#FC9272", "#FB6A4A", "#CB181D"],
    },
    {
        "field": "sl_pfmort",
        "title": "(c) Slum malaria mortality burden",
        "legend_title": "Mortality burden",
        "fmt": "deaths",
        "colors": ["#FFF5F0", "#FEE0D2", "#FCBBA1", "#FC9272", "#DE2D26", "#7F0000"],
    },
]

MALARIA_MAP = ["#FFF7F2", "#FEE5D9", "#FCBBA1", "#FC9272", "#FB6A4A", "#CB181D"]
TEXT = "#26343C"
BORDER = "#FFFFFF"
NA_COLOR = "#F2F2F2"
OCEAN = "#FFFFFF"

mpl.rcParams.update({
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.2,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})


def format_compact(value, fmt="people"):
    if value is None or not np.isfinite(value):
        return "NA"
    value = float(value)
    if value >= 1_000_000_000:
        return f"{value / 1_000_000_000:.1f}B"
    if value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    if value >= 1_000:
        return f"{value / 1_000:.0f}K"
    if fmt == "deaths" and value < 100:
        return f"{value:.1f}"
    return f"{value:.0f}"


def make_zero_positive_classes(values, n_classes=5):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = vals[vals >= 0]
    positive = vals[vals > 0]
    if positive.size == 0:
        return np.array([0, 1]), False
    cuts = np.quantile(positive, np.linspace(0, 1, n_classes + 1)[1:])
    edges = np.unique(np.r_[0, cuts])
    if edges.size < 2:
        edges = np.array([0, positive.max()])
    if edges[-1] <= edges[0]:
        edges[-1] = edges[0] + 1
    return edges, True


def legend_handles(edges, colors, fmt):
    handles = []
    if len(edges) >= 2:
        handles.append(Patch(facecolor=colors[0], edgecolor="none", label="0"))
    for idx in range(1, len(edges)):
        low = edges[idx - 1]
        high = edges[idx]
        if idx == 1:
            label = f">0 to {format_compact(high, fmt)}"
        else:
            label = f"{format_compact(low, fmt)} to {format_compact(high, fmt)}"
        handles.append(Patch(facecolor=colors[min(idx, len(colors)-1)], edgecolor="none", label=label))
    return handles


def setup_map_axis(ax, title):
    ax.set_facecolor(OCEAN)
    ax.set_title(title, loc="left", pad=5, color=TEXT, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_discrete_map(ax, gdf, field, title, legend_title, fmt, color_palette):
    setup_map_axis(ax, title)
    values = gdf[field].replace([np.inf, -np.inf], np.nan)
    edges, has_positive = make_zero_positive_classes(values)
    colors = color_palette[: max(len(edges), 2)]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(edges, cmap.N, clip=True)

    gdf.plot(ax=ax, color=NA_COLOR, edgecolor=BORDER, linewidth=0.10, zorder=1)
    if has_positive:
        gdf.assign(_plot_value=values.fillna(-1)).plot(
            column="_plot_value",
            ax=ax,
            cmap=cmap,
            norm=norm,
            edgecolor=BORDER,
            linewidth=0.10,
            missing_kwds={"color": NA_COLOR},
            zorder=2,
        )
        handles = legend_handles(edges, colors, fmt)
    else:
        handles = [Patch(facecolor=colors[0], edgecolor="none", label="0")]

    country_gdf.boundary.plot(
        ax=ax,
        edgecolor="#222222",
        linewidth=0.24,
        alpha=0.85,
        facecolor="none",
        zorder=3,
    )
    country_outer_boundary.boundary.plot(
        ax=ax,
        edgecolor="#111111",
        linewidth=0.48,
        facecolor="none",
        zorder=4,
    )

    leg = ax.legend(
        handles=handles,
        title=legend_title,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.02),
        frameon=False,
        facecolor="none",
        edgecolor="none",
        borderpad=0.42,
        labelspacing=0.32,
        handlelength=0.85,
        handleheight=0.52,
    )
    leg.get_title().set_fontsize(7.0)
    leg.set_frame_on(False)
    leg.get_frame().set_visible(False)
    ax.set_aspect("equal")


required_fields = [spec["field"] for spec in MAP_SPECS]
gdf = gpd.read_file(shp_path)
country_gdf = gpd.read_file(country_shp_path)
if country_gdf.crs != gdf.crs:
    country_gdf = country_gdf.to_crs(gdf.crs)
country_outer_boundary = country_gdf.dissolve()

missing = [field for field in required_fields if field not in gdf.columns]
if missing:
    raise KeyError(f"Missing required malaria fields: {missing}")
for field in required_fields:
    gdf[field] = gdf[field].replace([np.inf, -np.inf], np.nan)

fig, axes = plt.subplots(
    1, 3,
    figsize=(15.2, 5.2),
    dpi=300,
    gridspec_kw={"wspace": 0.02},
)

for ax, spec in zip(axes, MAP_SPECS):
    plot_discrete_map(
        ax, gdf,
        field=spec["field"],
        title=spec["title"],
        legend_title=spec["legend_title"],
        fmt=spec["fmt"],
        color_palette=spec["colors"],
    )

fig.savefig(output_png, dpi=600, bbox_inches="tight", pad_inches=0.08)
plt.show()


# malaria descriptive statistics

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd


admin_shp = Path(
    r"../data/overall/admin_Africa_with_RoofVul_SSA_HE6_flood_malaria.shp"
).resolve()

country_shp = Path(
    r"../data/country/country_SSA_HE2.shp"
).resolve()

out_dir = Path(r"tables").resolve()

out_dir.mkdir(parents=True, exist_ok=True)

admin_gdf = gpd.read_file(admin_shp)
country_gdf = gpd.read_file(country_shp)
if country_gdf.crs != admin_gdf.crs:
    country_gdf = country_gdf.to_crs(admin_gdf.crs)

# Attach country name and region by spatially joining each subnational unit's representative point.
admin_points = admin_gdf[["iso3", "geometry"]].copy()
admin_points["geometry"] = admin_points.geometry.representative_point()
country_lookup = gpd.sjoin(
    admin_points,
    country_gdf[["country_na", "region", "geometry"]],
    how="left",
    predicate="within",
)
country_lookup = country_lookup.loc[~country_lookup.index.duplicated(keep="first")]
admin_gdf["country_name"] = country_lookup["country_na"].reindex(admin_gdf.index).values
admin_gdf["region"] = country_lookup["region"].reindex(admin_gdf.index).values
fallback_country = {"COM": "Comoros", "STP": "Sao Tome and Principe", "SYC": "Seychelles"}
fallback_region = {"COM": "Eastern", "STP": "Central", "SYC": "Eastern"}
admin_gdf["country_name"] = admin_gdf["country_name"].fillna(admin_gdf["iso3"].map(fallback_country)).fillna(admin_gdf["iso3"])
admin_gdf["region"] = admin_gdf["region"].fillna(admin_gdf["iso3"].map(fallback_region)).fillna("Unassigned")

numeric_cols = [
    "sl_pop", "Heat_expos", "HI_expo", "sl_flpct", "sl_sevexp", "EDI_qmean",
    "sl_pfpop", "sl_pfinf", "sl_incpop", "sl_pfinc", "sl_morpop", "sl_pfmort",
]
for col in numeric_cols:
    admin_gdf[col] = pd.to_numeric(admin_gdf[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

# Reconstruct country flood totals from bounded per-unit indicators to avoid rounding/fallback artifacts
# where summed sl_flpop can slightly exceed summed sl_pop.
admin_gdf["flood_exposed_pop_reconstructed"] = admin_gdf["sl_pop"] * admin_gdf["sl_flpct"].clip(lower=0, upper=1)
admin_gdf["flood_severity_burden_reconstructed"] = admin_gdf["sl_pop"] * admin_gdf["sl_sevexp"].clip(lower=0)

def safe_divide(numerator, denominator):
    if pd.isna(numerator) or pd.isna(denominator) or denominator <= 0:
        return np.nan
    return numerator / denominator

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    valid = values.notna() & weights.notna() & (weights > 0)
    if valid.any():
        return float(np.average(values[valid], weights=weights[valid]))
    return float(values.mean()) if values.notna().any() else np.nan

records = []
for (iso3, country_name, region), df in admin_gdf.groupby(["iso3", "country_name", "region"], dropna=False):
    sl_pop = df["sl_pop"].sum(min_count=1)
    heat_total = df["Heat_expos"].sum(min_count=1)
    hi_total = df["HI_expo"].sum(min_count=1)
    flood_denominator = df.loc[df["sl_flpct"].notna(), "sl_pop"].sum(min_count=1)
    flood_pop = df["flood_exposed_pop_reconstructed"].sum(min_count=1)
    flood_severity_burden = df["flood_severity_burden_reconstructed"].sum(min_count=1)
    pf_pop = df["sl_pfpop"].sum(min_count=1)
    infection_burden = df["sl_pfinf"].sum(min_count=1)
    inc_pop = df["sl_incpop"].sum(min_count=1)
    incidence_burden = df["sl_pfinc"].sum(min_count=1)
    mort_pop = df["sl_morpop"].sum(min_count=1)
    mortality_burden = df["sl_pfmort"].sum(min_count=1)

    records.append({
        "region": region,
        "iso3": iso3,
        "country_name": country_name,
        "infection_burden": infection_burden,
        "incidence_burden": incidence_burden,
        "mortality_burden": mortality_burden,
        "incidence_per_infection": safe_divide(incidence_burden, infection_burden),
        "mortality_per_incidence": safe_divide(mortality_burden, incidence_burden),
    })

country_table = (
    pd.DataFrame(records)
    .sort_values(["region", "country_name", "iso3"], kind="mergesort")
    .reset_index(drop=True)
)

out_csv = out_dir / "malaria_country_descriptive_statistics.csv"
country_table.to_csv(out_csv, index=False, encoding="utf-8-sig")

display(country_table)

# figure4_Regional decomposition

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
SOURCE = Path(
    r"../figure4/result4_dual_amplification_source_data.csv"
).resolve()

OUT_DIR = Path(r"tables").resolve()

OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUT = OUT_DIR / "figure4_regional_decomposition_grouped_by_figure4_class.csv"

CLASS_ORDER = [
    "Dual amplification",
    "Low conversion",
    "Infection-to-incidence only",
    "Incidence-to-mortality only",
    "Other subnational units",
]
REGION_ORDER = ["Western", "Central", "Eastern", "Northern", "Southern"]
CONV_META = {
    "I2I": {
        "edi": "i2i_edi_contribution_sd",
        "heat": "i2i_heat_contribution_sd",
        "flood": "i2i_flood_contribution_sd",
        "joint": "i2i_joint_contribution_sd",
        "total": "i2i_mechanism_contribution_sd",
    },
    "I2M": {
        "edi": "i2m_edi_contribution_sd",
        "heat": "i2m_heat_contribution_sd",
        "flood": "i2m_flood_contribution_sd",
        "joint": "i2m_joint_contribution_sd",
        "total": "i2m_mechanism_contribution_sd",
    },
}


def load_source():
    df = pd.read_csv(SOURCE)
    df["sl_pop"] = pd.to_numeric(df["sl_pop"], errors="coerce").fillna(0).clip(lower=0)
    low_mask = df["clear_low_conversion"].fillna(False).astype(bool)
    df["figure4_class"] = "Other subnational units"
    df.loc[low_mask, "figure4_class"] = "Low conversion"
    for cls in ["Infection-to-incidence only", "Incidence-to-mortality only", "Dual amplification"]:
        df.loc[df["dual_class"].eq(cls), "figure4_class"] = cls

    contribution_cols = [meta[k] for meta in CONV_META.values() for k in ["edi", "heat", "flood", "joint", "total"]]
    for col in contribution_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def summarize_subset(sub, scope, region=None, denominator_pop=None):
    rows = []
    total_pop = sub["sl_pop"].sum() if denominator_pop is None else denominator_pop
    for cls in CLASS_ORDER:
        g = sub[sub["figure4_class"].eq(cls)].copy()
        n = int(len(g))
        pop = float(g["sl_pop"].sum())
        for conv, meta in CONV_META.items():
            rows.append(
                {
                    "scope": scope,
                    "region": region if region is not None else "All regions",
                    "figure4_class": cls,
                    "conversion": conv,
                    "admin_units_n": n,
                    "slum_population": pop,
                    "slum_population_million": pop / 1_000_000,
                    "slum_population_share": pop / total_pop if total_pop > 0 else np.nan,
                    "edi_contribution_mean_sd": float(g[meta["edi"]].mean()) if n else np.nan,
                    "heat_contribution_mean_sd": float(g[meta["heat"]].mean()) if n else np.nan,
                    "flood_contribution_mean_sd": float(g[meta["flood"]].mean()) if n else np.nan,
                    "heat_flood_joint_contribution_mean_sd": float(g[meta["joint"]].mean()) if n else np.nan,
                    "total_mechanism_contribution_mean_sd": float(g[meta["total"]].mean()) if n else np.nan,
                }
            )
    return rows


def build_summary(df):
    rows = []
    rows.extend(summarize_subset(df, scope="Overall", region="All regions"))
    for region in [r for r in REGION_ORDER if r in set(df["region"].dropna())]:
        sub = df[df["region"].eq(region)].copy()
        rows.extend(summarize_subset(sub, scope="Region", region=region, denominator_pop=sub["sl_pop"].sum()))
    summary = pd.DataFrame(rows)
    summary["figure4_class"] = pd.Categorical(summary["figure4_class"], CLASS_ORDER, ordered=True)
    summary["region"] = pd.Categorical(summary["region"], ["All regions", *REGION_ORDER], ordered=True)
    summary["conversion"] = pd.Categorical(summary["conversion"], ["I2I", "I2M"], ordered=True)
    return summary.sort_values(["scope", "figure4_class", "region", "conversion"]).reset_index(drop=True)


def display_table(summary, regional=False):
    d = summary.copy()
    d["Admin units, n"] = d["admin_units_n"].astype(int).astype(str)
    d["Slum population (million)"] = d["slum_population_million"].map(lambda x: "" if pd.isna(x) else f"{x:.1f}")
    d["Slum population share (%)"] = (d["slum_population_share"] * 100).map(
        lambda x: "" if pd.isna(x) else f"{x:.1f}"
    )
    for src, dst in [
        ("edi_contribution_mean_sd", "EDI"),
        ("heat_contribution_mean_sd", "Heat"),
        ("flood_contribution_mean_sd", "Flood"),
        ("heat_flood_joint_contribution_mean_sd", "Heat-flood"),
        ("total_mechanism_contribution_mean_sd", "Total"),
    ]:
        d[dst] = d[src].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
    columns = [
        "figure4_class",
        "conversion",
        "Admin units, n",
        "Slum population (million)",
        "Slum population share (%)",
        "EDI",
        "Heat",
        "Flood",
        "Heat-flood",
        "Total",
    ]
    rename = {"figure4_class": "Figure 4 class", "conversion": "Mechanism"}
    if regional:
        columns = ["figure4_class", "region", "conversion", "Admin units, n", "Slum population (million)",
                   "Slum population share (%)", "EDI", "Heat", "Flood", "Heat-flood", "Total"]
        rename["region"] = "Region"
    return d[columns].rename(columns=rename)


def main():
    df = load_source()
    summary = build_summary(df)
    regional = display_table(summary[summary["scope"].eq("Region")], regional=True)
    regional.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
    display(regional)


if __name__ == "__main__":
    main()
